# Setup & Dependencies

In [ ]:
%pip install -q --upgrade keras-cv
%pip install -q --upgrade keras  # Ensure Keras 3+
%pip install -q ultralytics kagglehub
%pip install -q pandas numpy tensorflow matplotlib scikit-learn opencv-python

import os
import shutil
import glob
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
import keras_cv
from keras_cv import bounding_box
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import kagglehub
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

# --- GLOBAL CONFIGURATION ---
IMG_SIZE = 320          # Increased for better detection
BATCH_SIZE = 16
EPOCHS = 40             # Enough for transfer learning
RANDOM_SEED = 42
NUM_CLASSES = 1         # Just 'hand'
CLASS_MAPPING = {0: "hand"}

# Paths
# Changed from "/content/hand_project" (Colab specific) to local directory
BASE_DIR = os.path.abspath("./hand_project")
RAW_DATA_DIR = f"{BASE_DIR}/raw"
WORK_DATA_DIR = f"{BASE_DIR}/processed"
MODELS_DIR = f"{BASE_DIR}/models"
YOLO_DIR = f"{BASE_DIR}/yolo_data"

for d in [BASE_DIR, RAW_DATA_DIR, WORK_DATA_DIR, MODELS_DIR, YOLO_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"✅ Setup Complete. Keras version: {keras.__version__}, KerasCV version: {keras_cv.__version__}")


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
ERROR: Operation cancelled by user
^C
ERROR: Operation cancelled by user
^C
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Data Download & Processing (Unified)

In [ ]:
# 1. DOWNLOAD DATA
def dataset_present():
    ann = os.path.join(RAW_DATA_DIR, "Annotations")
    return os.path.isdir(ann) and len(glob.glob(os.path.join(ann, "*.xml"))) > 0

if not dataset_present():
    print("⬇️ Downloading dataset...")
    cache_path = kagglehub.dataset_download("armannikkhah/hand-dataset")
    if os.path.exists(RAW_DATA_DIR): shutil.rmtree(RAW_DATA_DIR)
    shutil.copytree(cache_path, RAW_DATA_DIR)
    print("✅ Dataset downloaded.")
else:
    print("📦 Dataset already present.")

# 2. PARSE XML TO DATAFRAME
voc_root = None
for root, dirs, files in os.walk(RAW_DATA_DIR):
    if "Annotations" in dirs: voc_root = root; break

data = []
xml_files = glob.glob(os.path.join(voc_root, "Annotations", "*.xml"))
for xml_file in xml_files:
    root = ET.parse(xml_file).getroot()
    filename = root.find("filename").text
    img_path = os.path.join(voc_root, "JPEGImages", filename)
    
    if os.path.exists(img_path):
        w = int(root.find("size/width").text)
        h = int(root.find("size/height").text)
        
        # Collect all hands in image (detection can handle multiple)
        boxes = []
        for obj in root.findall("object"):
            if obj.find("name").text.lower() == "hand":
                b = obj.find("bndbox")
                # VOC format: xmin, ymin, xmax, ymax
                boxes.append([
                    float(b.find("xmin").text),
                    float(b.find("ymin").text),
                    float(b.find("xmax").text),
                    float(b.find("ymax").text)
                ])
        
        if boxes:
            # Store as path, width, height, boxes_list
            data.append({
                "image_path": img_path,
                "width": w, 
                "height": h,
                "boxes": boxes,
                "classes": [0] * len(boxes) # Class 0 for hand
            })

# Split Data
train_data, val_data = train_test_split(data, test_size=0.2, random_state=RANDOM_SEED)
print(f"✅ Data Processed. Train: {len(train_data)} | Val: {len(val_data)}")

# 3. PREPARE YOLO FILES (YOLO needs text files on disk)
def create_yolo_files(dataset_list, split_name):
    img_dir = f"{YOLO_DIR}/images/{split_name}"
    lbl_dir = f"{YOLO_DIR}/labels/{split_name}"
    os.makedirs(img_dir, exist_ok=True); os.makedirs(lbl_dir, exist_ok=True)
    
    for item in dataset_list:
        # Copy image
        shutil.copy(item['image_path'], f"{img_dir}/{os.path.basename(item['image_path'])}")
        
        # Write label file (class x_center y_center w h - normalized)
        txt_path = f"{lbl_dir}/{os.path.splitext(os.path.basename(item['image_path']))[0]}.txt"
        with open(txt_path, "w") as f:
            W, H = item['width'], item['height']
            for box in item['boxes']:
                xmin, ymin, xmax, ymax = box
                # Convert to center_x, center_y, w, h
                cx = ((xmin + xmax) / 2) / W
                cy = ((ymin + ymax) / 2) / H
                bw = (xmax - xmin) / W
                bh = (ymax - ymin) / H
                f.write(f"0 {cx} {cy} {bw} {bh}\n")

print("Creating YOLO files...")
create_yolo_files(train_data, "train")
create_yolo_files(val_data, "val")

# Write YAML
with open(f"{YOLO_DIR}/data.yaml", "w") as f:
    f.write(f"path: {YOLO_DIR}\ntrain: images/train\nval: images/val\nnames:\n  0: hand")

# KerasCV Data Loader Pipeline

In [ ]:
# Loader for KerasCV
def load_dict(item):
    # Load image
    img = tf.io.read_file(item['image_path'])
    img = tf.io.decode_jpeg(img, channels=3)
    
    # Bounding boxes need to be in a dict structure
    bounding_boxes = {
        "classes": tf.cast(item['classes'], dtype=tf.float32),
        "boxes": tf.cast(item['boxes'], dtype=tf.float32), # Format: xyxy
    }
    return {"images": img, "bounding_boxes": bounding_boxes}

def build_dataset(data_list, batch_size=BATCH_SIZE, training=True):
    # Convert list of dicts to tf.data
    # We need to "ragged" batching usually because number of boxes varies
    # But for KerasCV, we often use fixed padding.
    
    # Create generator
    def gen():
        for item in data_list:
            yield load_dict(item)

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature={
            "images": tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
            "bounding_boxes": {
                "classes": tf.TensorSpec(shape=(None,), dtype=tf.float32),
                "boxes": tf.TensorSpec(shape=(None, 4), dtype=tf.float32),
            }
        }
    )

    if training:
        ds = ds.shuffle(100)

    # Preprocessing Layer (Resizing & Padding)
    # KerasCV models expect inputs to be resized/padded
    augmenter = keras_cv.layers.JitteredResize(
        target_size=(IMG_SIZE, IMG_SIZE), 
        scale_factor=(0.8, 1.25), 
        bounding_box_format="xyxy"
    )
    
    # Inference resizer (no jitter)
    resizer = keras_cv.layers.Resizing(
        height=IMG_SIZE, width=IMG_SIZE, bounding_box_format="xyxy", pad_to_aspect_ratio=True
    )

    ds = ds.map(lambda x: (x['images'], x['bounding_boxes']), num_parallel_calls=tf.data.AUTOTUNE)
    
    # Batch first (needed for variable number of boxes before padding)
    # Actually, KerasCV recommends creating a RaggedTensor batch then mapping
    ds = ds.ragged_batch(batch_size, drop_remainder=training)
    
    # Apply resizing/aug
    if training:
        ds = ds.map(augmenter, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(resizer, num_parallel_calls=tf.data.AUTOTUNE)
        
    # Convert ragged to dense (pad boxes)
    # This ensures every batch has shape (B, N, 4)
    ds = ds.map(
        lambda img, boxes: (
            img, 
            bounding_box.to_dense(boxes, max_boxes=20) # Max 20 hands per image
        ), 
        num_parallel_calls=tf.data.AUTOTUNE
    )
    
    # Standardize inputs (0-1 or similar depends on model, but usually KerasCV handles this inside model presets. 
    # We will just cast to float32 here)
    ds = ds.map(lambda img, boxes: (tf.cast(img, tf.float32), boxes), num_parallel_calls=tf.data.AUTOTUNE)

    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = build_dataset(train_data, training=True)
val_ds = build_dataset(val_data, training=False)

# Check a batch
img_batch, box_batch = next(iter(train_ds))
print(f"Batch Image Shape: {img_batch.shape}")
print(f"Batch Boxes Shape: {box_batch['boxes'].shape}")

# Model 1 - YOLOv8 Nano (The Baseline)

In [ ]:
print(f"🚀 Training YOLOv8 Nano...")
model_yolo = YOLO("yolov8n.pt")
model_yolo.train(
    data=f"{YOLO_DIR}/data.yaml",
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=f"{MODELS_DIR}/yolo_runs",
    name="hand_yolo",
    exist_ok=True,
    verbose=False
)
print("✅ YOLOv8 Trained.")

# Model 2 - RetinaNet (KerasCV)

In [ ]:
# Build RetinaNet
# We use a ResNet50 backbone pre-trained on Coco
model_retinanet = keras_cv.models.RetinaNet.from_preset(
    "resnet50_imagenet",
    num_classes=1,
    bounding_box_format="xyxy"
)

# Optimization
# RetinaNet can be tricky with LR. Using a schedule helps.
lr_decay = keras.optimizers.schedules.PiecewiseConstantDecay(
    boundaries=[1000], values=[0.005, 0.001]
)
optimizer = keras.optimizers.SGD(learning_rate=lr_decay, momentum=0.9, global_clipnorm=10.0)

model_retinanet.compile(
    classification_loss="focal",
    box_loss="smooth_l1",
    optimizer=optimizer,
    jit_compile=False
)

print("🚀 Training RetinaNet (ResNet50)...")
hist_retina = model_retinanet.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        keras.callbacks.ModelCheckpoint(f"{MODELS_DIR}/retinanet_best.keras", save_best_only=True)
    ]
)

# Model 3 - SSD MobileNet (KerasCV)

In [ ]:
print(f"🚀 Training YOLOv8 SMALL (Larger Model)...")
model_yolo_s = YOLO("yolov8s.pt") # Small version, bigger than Nano
model_yolo_s.train(
    data=f"{YOLO_DIR}/data.yaml",
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=f"{MODELS_DIR}/yolo_runs",
    name="hand_yolo_small",
    exist_ok=True,
    verbose=False
)
print("✅ YOLOv8 Small Trained.")

# Unified Evaluation & Visualization

In [ ]:
# Load Best Models
yolo_n = YOLO(f"{MODELS_DIR}/yolo_runs/hand_yolo/weights/best.pt")
yolo_s = YOLO(f"{MODELS_DIR}/yolo_runs/hand_yolo_small/weights/best.pt")
# RetinaNet is loaded from memory or file
# model_retinanet.load_weights(f"{MODELS_DIR}/retinanet_best.keras")

# Helper to get boxes
def get_yolo_boxes(model, img_path):
    res = model.predict(img_path, imgsz=IMG_SIZE, verbose=False, conf=0.25)[0]
    return res.boxes.xyxy.cpu().numpy()

def get_keras_boxes(model, img_path):
    # Preprocess
    img = tf.io.read_file(img_path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.expand_dims(img, 0)
    
    # Predict
    pred = model.predict(img, verbose=0)
    # Parse (KerasCV output is dict)
    # We need to filter by confidence, usually checks 'confidence' key or similar depending on version
    # Simplified:
    boxes = pred['boxes'][0]
    scores = pred['confidence'][0]
    
    # Filter
    valid_indices = tf.where(scores > 0.25)
    valid_boxes = tf.gather(boxes, valid_indices)
    
    # Scale boxes back to original size (tricky because we resized with padding)
    # For viz, we will just visualize on the resized image 320x320
    return valid_boxes.numpy().squeeze()

# Visualization
import cv2

def draw_boxes(img, boxes, color=(0, 255, 0)):
    for box in boxes:
        if len(box) < 4: continue
        x1, y1, x2, y2 = map(int, box[:4])
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    return img

# Sample 3 images
sample_imgs = val_data[:3]

plt.figure(figsize=(15, 10))

for i, item in enumerate(sample_imgs):
    path = item['image_path']
    
    # Load Original for drawing
    orig = cv2.imread(path)
    orig = cv2.cvtColor(orig, cv2.COLOR_BGR2RGB)
    orig = cv2.resize(orig, (IMG_SIZE, IMG_SIZE)) # Resize for consistency in viz
    
    # 1. YOLO Nano
    yn_boxes = get_yolo_boxes(yolo_n, path) 
    # Note: YOLO predicts on resized but outputs coords relative to original. 
    # We need to scale them to 320 for this viz or plot on original.
    # Let's stick to plotting on 320 resized for simplicity
    scale_x = IMG_SIZE / item['width']
    scale_y = IMG_SIZE / item['height']
    yn_boxes_scaled = yn_boxes.copy()
    yn_boxes_scaled[:, [0,2]] *= scale_x
    yn_boxes_scaled[:, [1,3]] *= scale_y
    
    img1 = draw_boxes(orig.copy(), yn_boxes_scaled, (255, 0, 0)) # Red
    
    # 2. YOLO Small
    ys_boxes = get_yolo_boxes(yolo_s, path)
    ys_boxes_scaled = ys_boxes.copy()
    ys_boxes_scaled[:, [0,2]] *= scale_x
    ys_boxes_scaled[:, [1,3]] *= scale_y
    img2 = draw_boxes(orig.copy(), ys_boxes_scaled, (0, 255, 0)) # Green
    
    # 3. RetinaNet
    # RetinaNet prediction is already on 320x320 space from our preproc
    rn_boxes = get_keras_boxes(model_retinanet, path)
    img3 = draw_boxes(orig.copy(), rn_boxes, (0, 0, 255)) # Blue

    # Plot
    plt.subplot(3, 3, i*3 + 1); plt.imshow(img1); plt.title("YOLOv8 Nano")
    plt.subplot(3, 3, i*3 + 2); plt.imshow(img2); plt.title("YOLOv8 Small")
    plt.subplot(3, 3, i*3 + 3); plt.imshow(img3); plt.title("RetinaNet")

plt.tight_layout()
plt.show()